# Lab: A Car Rental Agent with Sessions + Long-Term Memory

In this lab you build a small **car rental assistant** with **LangGraph** and a **Unity Catalog hosted model**, then give it two kinds of memory with **Databricks Agent Memory** (backed by Lakebase):

| Memory type | What it does | Scope | In this lab |
|---|---|---|---|
| **Session checkpointing** (short-term) | Saves the conversation state after each turn, so the agent can resume a conversation | One `session_id` (conversation) | LangGraph `checkpointer` backed by a **session store** |
| **Long-term memory** | Stores durable facts about a customer that carry across conversations | One `actor_id` (user) | `memory_tools` + recall, backed by a **memory store** |

```
   user message ──► LangGraph agent ──► UC hosted LLM
                      │      │
     checkpointer ◄───┘      └───► memory tools (recall / remember / forget / list_memories)
          │                                  │
  Session store (per conversation)   Memory store (per customer)
          └──────────── Lakebase ────────────┘
```

**What you will do**
1. Install libraries and configure names
2. Create a session store and a memory store
3. Build a simple car rental agent (3 tools over a fake fleet)
4. See **session checkpointing**: the agent remembers earlier turns in the same conversation
5. See **long-term memory**: the agent remembers a customer's preferences in a brand-new conversation
6. Inspect what was stored, then clean up

Based on the Databricks tutorial [Add sessions and memory to an existing agent](https://docs.databricks.com/aws/en/agents/agent-memory/existing-agent-tutorial?language=LangGraph).

**Requirements:** a workspace with Agent Memory (Beta) enabled, serverless or DBR compute with Python 3.10+, and permission to query a Foundation Model serving endpoint.

## Step 1: Install libraries

- `databricks-agentbricks` contains the AgentKit SDK (`databricks_agentkit`) with the session and memory stores, plus the LangGraph adapters.
- The `[langgraph]` extra also installs LangGraph, LangChain, and `databricks-langchain` (which gives us `ChatDatabricks` to call a Unity Catalog hosted model).
- Versions are pinned so pip resolves quickly.

In [0]:
%pip install "databricks-agentbricks[langgraph]==0.6.0" "databricks-langchain==0.21.0" "databricks-openai==0.17.1"

In [0]:
# Restart Python so the newly installed packages are loaded
dbutils.library.restartPython()

## Step 2: Configure the lab

Pick the model endpoint and a short prefix so your stores don't collide with other lab participants.
Store IDs must be **lowercase, 3-56 characters**.

> To see available models, open **Serving** in the left nav, or browse **Catalog > system > ai**. Any chat model that supports tool calling works (for example a Claude, GPT, or Llama endpoint).

In [0]:
dbutils.widgets.text("model_endpoint", "databricks-claude-opus-4-5", "Model serving endpoint")
dbutils.widgets.text("prefix", "carlab", "Store name prefix")

import re

MODEL_ENDPOINT = dbutils.widgets.get("model_endpoint")
current_user = spark.sql("SELECT current_user()").first()[0]
user_slug = re.sub(r"[^a-z0-9]+", "-", current_user.split("@")[0].lower()).strip("-")
prefix = dbutils.widgets.get("prefix").lower()

SESSION_STORE = f"{prefix}-{user_slug}-sessions"[:56]
MEMORY_STORE = f"{prefix}-{user_slug}-memory"[:56]

print("Model endpoint:", MODEL_ENDPOINT)
print("Session store: ", SESSION_STORE)
print("Memory store:  ", MEMORY_STORE)

## Step 3: Create the session store and memory store

Both stores are **workspace-scoped** and are created once, then reused by every run of the agent.

- **Session store**: holds conversation checkpoints, keyed by `session_id`.
- **Memory store**: holds long-term facts, keyed by `actor_id` (the customer).

The cell below creates each store only if it doesn't exist yet, so it's safe to re-run.

> **Where is the data?** You don't create a database yourself. The first time you create a store, Databricks provisions a managed
> Lakebase Autoscaling project named **`databricks-internal-custom-agents`**, and each store gets its own Postgres database in it
> (`session-<store>` and `memory-<store>`). You can see it under **Compute > Lakebase**. The first store creation can take a minute or two.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks_agentkit import AgentKitClient

client = AgentKitClient(WorkspaceClient())


def get_or_create(stores, name, description):
    try:
        return stores.get(name)
    except Exception:
        print(f"Creating {name} ...")
        return stores.create(name, description=description)


session_store = get_or_create(client.session_stores, SESSION_STORE, "Car rental lab - conversations")
memory_store = get_or_create(client.memory_stores, MEMORY_STORE, "Car rental lab - customer memory")
print("Stores ready.")

## Step 4: Build the car rental tools

To keep things simple the "fleet" is a Python dictionary. The agent gets three tools:

| Tool | Purpose |
|---|---|
| `search_cars` | Find available cars in a city, optionally by type |
| `quote_price` | Price a rental for a number of days (applies loyalty discounts) |
| `book_car` | Make a (fake) reservation |

In [0]:
from langchain_core.tools import tool

FLEET = {
    "SEA-01": {"city": "Seattle", "type": "SUV",     "model": "Toyota RAV4",     "daily_rate": 75},
    "SEA-02": {"city": "Seattle", "type": "Compact", "model": "Honda Civic",     "daily_rate": 45},
    "SEA-03": {"city": "Seattle", "type": "EV",      "model": "Tesla Model 3",   "daily_rate": 90},
    "DEN-01": {"city": "Denver",  "type": "SUV",     "model": "Subaru Outback",  "daily_rate": 70},
    "DEN-02": {"city": "Denver",  "type": "Compact", "model": "Toyota Corolla",  "daily_rate": 40},
    "SFO-01": {"city": "San Francisco", "type": "EV",  "model": "Chevy Bolt",    "daily_rate": 65},
    "SFO-02": {"city": "San Francisco", "type": "SUV", "model": "Ford Explorer", "daily_rate": 85},
}
LOYALTY_DISCOUNT = {"gold": 0.15, "silver": 0.10, "none": 0.0}


@tool
def search_cars(city: str, car_type: str = "") -> str:
    """Search available rental cars in a city. car_type is optional: SUV, Compact, or EV."""
    matches = [
        f"{car_id}: {c['model']} ({c['type']}) - ${c['daily_rate']}/day"
        for car_id, c in FLEET.items()
        if c["city"].lower() == city.lower() and (not car_type or c["type"].lower() == car_type.lower())
    ]
    return "\n".join(matches) or f"No cars found in {city}."


@tool
def quote_price(car_id: str, days: int, loyalty_tier: str = "none") -> str:
    """Quote the total rental price. loyalty_tier is gold, silver, or none."""
    car = FLEET.get(car_id.upper())
    if not car:
        return f"Unknown car id {car_id}."
    discount = LOYALTY_DISCOUNT.get(loyalty_tier.lower(), 0.0)
    total = car["daily_rate"] * days * (1 - discount)
    return f"{car['model']} for {days} days: ${total:.2f} (discount {discount:.0%})"


@tool
def book_car(car_id: str, days: int, customer_name: str) -> str:
    """Book a rental car for a customer."""
    car = FLEET.get(car_id.upper())
    if not car:
        return f"Unknown car id {car_id}."
    return f"Confirmed: {car['model']} in {car['city']} for {days} days, booked for {customer_name}. Ref RNT-{car_id.upper()}-{days}"


rental_tools = [search_cars, quote_price, book_car]
print(search_cars.invoke({"city": "Seattle"}))

## Step 5: Build the agent with sessions + memory

Three pieces from `databricks_agentkit.langgraph` add memory to a regular LangGraph agent:

1. **`checkpointer(store=SESSION_STORE)`** – LangGraph saves the full conversation state to the session store after every step. Re-using the same `session_id` resumes the conversation.
2. **`thread_config(session_id, actor=user_id)`** – the per-call config that tells the checkpointer *which* conversation and *which* user.
3. **`memory_tools(user_id, store=MEMORY_STORE)`** – gives the model `recall`, `remember`, `forget`, and `list_memories` tools, **scoped to one customer**.

We also **recall relevant facts before each turn** and add them to the system prompt, so the agent doesn't have to remember to call `recall` itself.

Each memory has a **path** (like `/loyalty.md`), which is its name for that customer. Saving to the same path replaces the old fact.
The `remember` tool takes a **topic** and turns it into the path (topic `loyalty` → `/loyalty.md`). The prompt gives the model a fixed
set of topics, so facts get updated instead of duplicated. The `.md` is only a naming convention: paths and content are both stored as rows in Lakebase.

> **Security note:** `user_id` (the `actor_id`) must always come from trusted application code (for example the signed-in user), never from the model or the chat text.

In [0]:
import warnings
from databricks_langchain import ChatDatabricks
from langchain.agents import create_agent
from databricks_agentkit.langgraph import checkpointer, memory_tools, thread_config

model = ChatDatabricks(endpoint=MODEL_ENDPOINT)
warnings.filterwarnings("ignore", message="Pydantic serializer warnings")  # harmless noise from reasoning blocks

BASE_PROMPT = """You are a friendly assistant for Lakehouse Car Rentals.
Help customers find, price, and book rental cars using your tools. Keep answers short.
When a customer shares a durable preference or fact about themselves (favorite car type,
loyalty tier, name, home city, accessibility needs), save it with the `remember` tool.
Do not save one-off trip details like dates.
Always use one of these topics when you save a memory, replacing the old fact if it changed:
  profile      - name, home city, accessibility needs
  preferences  - car preferences
  loyalty      - loyalty tier"""


def recalled_facts(user_id: str, message: str) -> str:
    """Search long-term memory for facts relevant to this message."""
    results = memory_store.search(actor_id=user_id, query=message, limit=5)
    return "\n".join(f"- {r.memory.content}" for r in results)


async def chat(user_id: str, session_id: str, message: str) -> str:
    """Run one turn of the conversation for a given customer and session."""
    system_prompt = BASE_PROMPT
    facts = recalled_facts(user_id, message)
    if facts:
        system_prompt += f"\n\nWhat you know about this customer:\n{facts}"

    agent = create_agent(
        model=model,
        tools=[*rental_tools, *memory_tools(user_id, store=MEMORY_STORE)],
        system_prompt=system_prompt,
        checkpointer=checkpointer(store=SESSION_STORE),
    )
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": message}]},
        config=thread_config(session_id, actor=user_id),
    )
    reply = result["messages"][-1].text  # .text keeps only the answer text, without reasoning blocks
    print(f"[{user_id} | {session_id}]\nCustomer: {message}\nAgent:    {reply}\n")
    return reply

## Step 6: Session checkpointing (short-term memory)

Alice starts a conversation (`session_id = "trip-denver-..."`). Notice the second message never mentions Denver -
the agent knows because the conversation history was **checkpointed** and reloaded for the second turn.

Long-term memories survive between lab runs, so we first wipe Alice's and Bob's memories to start the demo clean.

In [0]:
import uuid

ALICE = "alice@example.com"
BOB = "bob@example.com"
run_id = uuid.uuid4().hex[:6]  # keeps session ids unique if you re-run the lab
denver_session = f"trip-denver-{run_id}"

# Start the demo with no long-term memories for our two customers
for user in (ALICE, BOB):
    for m in memory_store.list(actor_id=user):
        m.delete()

await chat(ALICE, denver_session, "Hi, what cars do you have in Denver?")
await chat(ALICE, denver_session, "How much is the cheapest one for 4 days?")

### Resume vs. start fresh

- **Same `session_id`** → the agent resumes. Even if the notebook restarted, the state lives in the session store, not in Python memory.
- **New `session_id`** → a blank conversation. The agent has no idea we were talking about Denver.

In [0]:
await chat(ALICE, denver_session, "Great, book it for Alice.")

await chat(ALICE, f"new-chat-{run_id}", "Which car did we just talk about?")

## Step 7: Long-term memory (across sessions)

Session checkpoints are per conversation. To remember things **about the customer** across conversations, we use the memory store.

Alice shares two durable facts. The agent should call its `remember` tool to save them.

In [0]:
await chat(
    ALICE,
    f"profile-chat-{run_id}",
    "By the way, I always prefer SUVs, and I'm a Gold loyalty member. Please remember that.",
)

### Brand-new conversation, different city

Alice starts a **new session** and doesn't repeat her preferences. Because `chat()` recalls facts from the memory store,
the agent should suggest an **SUV** and apply the **Gold discount** automatically.

In [0]:
await chat(ALICE, f"trip-seattle-{run_id}", "I need a car in Seattle for 3 days. What do you recommend and what will it cost?")

### Memory is per customer

Bob is a different `actor_id`, so he gets none of Alice's memories - no SUV bias and no Gold discount.

> **Watch out:** a session is looked up by `session_id`. Always give each conversation its own unique ID (in an app, a fresh UUID).
> If Bob reused Alice's `session_id`, he would resume *her* conversation.

In [0]:
await chat(BOB, f"bob-trip-seattle-{run_id}", "I need a car in Seattle for 3 days. What do you recommend and what will it cost?")

## Step 8: Inspect what's stored

Long-term memories are regular entries you can list, search, and delete with the SDK - useful for debugging,
or for building a "what do you know about me?" page.

In [0]:
for m in memory_store.list(actor_id=ALICE):
    print(f"{m.path}: {m.content}")

print("\nBob's memories:", list(memory_store.list(actor_id=BOB)))

You can also write memories directly from application code (for example, loading the loyalty tier from your CRM)
instead of waiting for the model to learn it:

In [0]:
memory_store.put(
    actor_id=BOB,
    path="/loyalty.md",  # same path the agent uses, so later updates replace it
    content="Bob is a Silver loyalty member.",
    description="Loyalty tier from CRM",
)

await chat(BOB, f"trip-sfo-{run_id}", "Price an EV in San Francisco for 2 days, please.")

## Step 9 (optional): Try it yourself

1. Tell the agent *"Actually, I'm not a Gold member anymore"* and check whether it uses `forget` or updates the memory. Re-run the Step 8 listing.
2. Ask *"What do you remember about me?"* - the agent can call `list_memories`.
3. Add a new preference (e.g. *"I need a car seat for my toddler"*) and see it recalled in a new session.

**Going further:** instead of relying on the `remember` tool, a **memory pipeline** can extract memories from past sessions automatically
(`agentbricks memory pipeline create ...`). See [Agent memory pipelines](https://docs.databricks.com/aws/en/agents/agent-memory/memory-pipelines).

In [0]:
# Your experiments here
await chat(ALICE, f"recap-{run_id}", "What do you remember about me?")

## Step 10: Clean up

Stores are backed by Lakebase. Uncomment and run to delete the lab memories, or delete the stores entirely from a terminal:

```bash
agentbricks memory stores delete <your-memory-store>
```

In [0]:
# for user in (ALICE, BOB):
#     for m in memory_store.list(actor_id=user):
#         m.delete()
# print("Lab memories deleted.")

## Recap

| | Session checkpointing | Long-term memory |
|---|---|---|
| Key | `session_id` | `actor_id` |
| Remembers | The whole conversation | Durable facts about the customer |
| LangGraph hook | `checkpointer(store=...)` + `thread_config(...)` | `memory_tools(...)` + recall in the system prompt |
| Example | "How much is *the cheapest one*?" | "Alice prefers SUVs and is Gold" |

A few lines of code turned a stateless LangGraph agent into one that resumes conversations and gets to know its customers, with all state stored in Lakebase.